# Qwen 3.5 LiteRT + Task Bundle (Kaggle)

This notebook does **two things**:
1. Converts the Qwen 3.5 model to LiteRT (`.tflite`) format.
2. Bundles the `.tflite` + Qwen tokenizer into a MediaPipe `.task` file for Android.

**Run this on Kaggle** (30GB RAM required).

In [ ]:
# ============================================================
# CELL 1: Clone the repository and install dependencies
# ============================================================
import os

WORK_DIR = '/kaggle/working/litert-torch-qwen3.5'

if not os.path.exists(WORK_DIR):
    !git clone https://github.com/byepv2493k-ship-it/litert-torch-qwen3.5.git {WORK_DIR}

%cd {WORK_DIR}

# Install core LiteRT / AI Edge packages
!pip install ai-edge-litert-nightly==2.2.0.dev20260319 ai-edge-quantizer-nightly==0.5.1.dev20260320

# Install torchao nightly
!pip install --extra-index-url https://download.pytorch.org/whl/nightly/cpu torchao==0.16.0

# Install litert_torch from this repo
!pip install -e .

# Install TensorFlow + PyTorch nightly
!pip install tensorflow==2.21.0
!pip install --extra-index-url https://download.pytorch.org/whl/nightly/cpu torch==2.12.0.dev20260320+cpu

# Install HuggingFace + other deps
!pip install huggingface_hub==1.7.2 transformers==5.3.0 safetensors==0.7.0 sentencepiece==0.2.1 tokenizers==0.22.2
!pip install numpy==2.4.3 scipy==1.17.1 jax==0.9.2 jaxlib==0.9.2 ml_dtypes==0.5.4
!pip install absl-py fire flatbuffers protobuf PyYAML Jinja2 sympy tqdm regex pillow

# *** NEW: Install MediaPipe for the bundler ***
!pip install mediapipe

In [ ]:
# ============================================================
# CELL 2: Download the Qwen Model from HuggingFace
# ============================================================
import os
from huggingface_hub import snapshot_download

# Option A: Set token directly
os.environ['HF_TOKEN'] = 'put_your_hf_token_here'

# Option B: Use Kaggle Secrets (recommended)
# from kaggle_secrets import UserSecretsClient
# os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

model_path = snapshot_download(
    'Qwen/Qwen3.5-0.8B',
    token=os.environ['HF_TOKEN'], 
    ignore_patterns=['*.bin','*.msgpack','*.onnx']
)
print(f'Model downloaded to: {model_path}')

In [ ]:
# ============================================================
# CELL 3: Patch mmap_utils (version mismatch fix)
# ============================================================
import ai_edge_litert.tools.mmap_utils as _mu
if not hasattr(_mu, 'advise_dont_need'):
    _mu.advise_dont_need = lambda *a, **kw: None
    print('Patched mmap_utils.advise_dont_need')
else:
    print('mmap_utils.advise_dont_need already exists')

In [ ]:
# ============================================================
# CELL 4: Run the TFLite Converter
# ============================================================
OUTPUT_DIR = '/kaggle/working/output'
!mkdir -p {OUTPUT_DIR}

!python -m litert_torch.generative.examples.qwen.convert_v3_5_to_tflite \
    --checkpoint_path={model_path} \
    --output_path={OUTPUT_DIR} \
    --model_size=0.8b \
    --quantize=none \
    --kv_cache_max_len=2048 \
    --prefill_seq_lens=8 \
    --prefill_seq_lens=64 \
    --prefill_seq_lens=128 \
    --prefill_seq_lens=256 \
    --prefill_seq_lens=512

print('TFLite conversion finished!')

In [ ]:
# ============================================================
# CELL 5: Convert Qwen Tokenizer to SentencePiece format
# ============================================================
import os
SPM_OUTPUT = os.path.join(OUTPUT_DIR, 'tokenizer.spm')

!python -m litert_torch.generative.tools.tokenizer_to_sentencepiece \
    --checkpoint={model_path} \
    --output_path={SPM_OUTPUT} \
    --normalize_tokens=decode

print(f'SentencePiece tokenizer saved to: {SPM_OUTPUT}')

In [ ]:
# ============================================================
# CELL 6: Bundle .tflite + tokenizer into a .task file
# ============================================================
import glob
import os

# Find the generated .tflite file
tflite_files = glob.glob(os.path.join(OUTPUT_DIR, '*.tflite'))
print(f'Found TFLite files: {tflite_files}')

# Use the largest .tflite file
tflite_path = sorted(tflite_files, key=os.path.getsize, reverse=True)[0]
print(f'Using: {tflite_path}')

TASK_OUTPUT = os.path.join(OUTPUT_DIR, 'qwen3_5_0_8b.task')

from mediapipe.tasks.python.genai import bundler

config = bundler.BundleConfig(
    tflite_model=tflite_path,
    tokenizer_model=SPM_OUTPUT,
    start_token='<|im_start|>',
    stop_tokens=['<|im_end|>', '<|endoftext|>'],
    output_filename=TASK_OUTPUT,
)
bundler.create_bundle(config)

print(f'Bundle created at: {TASK_OUTPUT}')